In [2]:
#r "task14/bin/Debug/net10.0/task14.dll"
#r "nuget:ScottPlot,5.0.54"
#r "nuget:xunit,2.9.2"

using System;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task14;

Console.WriteLine("=== Определение оптимального шага ===\n");

double a = -100.0;
double b = 100.0;
double targetPrecision = 1e-4;
var steps = new double[] { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
int testThreads = 4;
int warmupRuns = 2;
int measureRuns = 5;

double exactValue = 0.0;

// Вспомогательная функция для форматирования шага
string FormatStep(double step)
{
    int exponent = (int)Math.Log10(step);
    return $"1e{exponent}";
}

var stepResults = new (double step, double error, double singleTime, double multiTime, double percentFaster)[steps.Length];

for (int i = 0; i < steps.Length; i++)
{
    double step = steps[i];
    
    double result = DefiniteIntegral.SolveSingleThread(a, b, Math.Sin, step);
    double error = Math.Abs(result - exactValue);
    
    double singleTotalTime = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var sw = Stopwatch.StartNew();
        DefiniteIntegral.SolveSingleThread(a, b, Math.Sin, step);
        sw.Stop();
        if (run >= warmupRuns)
            singleTotalTime += sw.Elapsed.TotalMilliseconds;
    }
    double singleAvgTime = singleTotalTime / measureRuns;
    
    double multiTotalTime = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var sw = Stopwatch.StartNew();
        DefiniteIntegral.Solve(a, b, Math.Sin, step, testThreads);
        sw.Stop();
        if (run >= warmupRuns)
            multiTotalTime += sw.Elapsed.TotalMilliseconds;
    }
    double multiAvgTime = multiTotalTime / measureRuns;
    
    double percentFaster = (singleAvgTime - multiAvgTime) / singleAvgTime * 100.0;
    
    stepResults[i] = (step, error, singleAvgTime, multiAvgTime, percentFaster);
    Console.WriteLine($"Шаг: {FormatStep(step),6} | Точность: {error,10:F6} | Один поток: {singleAvgTime,8:F2} мс | Много: {multiAvgTime,8:F2} мс | Выигрыш: {percentFaster,6:F1}%");
}

var validSteps = stepResults
    .Where(r => r.error <= targetPrecision && r.percentFaster >= 15.0)
    .ToArray();

if (validSteps.Length == 0)
{
    Console.WriteLine("\nНет шага с выигрышем >= 15%! Выбираем шаг с максимальным выигрышем среди точных.");
    validSteps = stepResults
        .Where(r => r.error <= targetPrecision)
        .OrderByDescending(r => r.percentFaster)
        .ToArray();
}

var optimalStep = validSteps
    .OrderBy(r => r.multiTime)
    .First();

Console.WriteLine($"\nОптимальный шаг: {FormatStep(optimalStep.step)}");
Console.WriteLine($"  Погрешность: {optimalStep.error:F6}");
Console.WriteLine($"  Однопоточное время: {optimalStep.singleTime:F2} мс");
Console.WriteLine($"  Многопоточное время: {optimalStep.multiTime:F2} мс");
Console.WriteLine($"  Выигрыш: {optimalStep.percentFaster:F1}%\n");

Console.WriteLine("=== Определение оптимального числа потоков ===\n");

int maxThreads = Environment.ProcessorCount * 2;
var threadCounts = Enumerable.Range(1, maxThreads).ToArray();
var threadResults = new (int threads, double avgTime, double percentFaster)[threadCounts.Length];

double singleTimeForStep = 0;
for (int run = 0; run < warmupRuns + measureRuns; run++)
{
    var sw = Stopwatch.StartNew();
    DefiniteIntegral.SolveSingleThread(a, b, Math.Sin, optimalStep.step);
    sw.Stop();
    if (run >= warmupRuns)
        singleTimeForStep += sw.Elapsed.TotalMilliseconds;
}
singleTimeForStep /= measureRuns;

for (int i = 0; i < threadCounts.Length; i++)
{
    int threads = threadCounts[i];
    
    double totalTime = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        var sw = Stopwatch.StartNew();
        DefiniteIntegral.Solve(a, b, Math.Sin, optimalStep.step, threads);
        sw.Stop();
        if (run >= warmupRuns)
            totalTime += sw.Elapsed.TotalMilliseconds;
    }
    double avgTime = totalTime / measureRuns;
    double percentFaster = (singleTimeForStep - avgTime) / singleTimeForStep * 100.0;
    
    threadResults[i] = (threads, avgTime, percentFaster);
    Console.WriteLine($"Потоков: {threads,2} | Время: {avgTime,8:F2} мс | Выигрыш: {percentFaster,6:F1}%");
}

var optimalThreads = threadResults.OrderByDescending(r => r.percentFaster).First();
Console.WriteLine($"\nОптимальное число потоков: {optimalThreads.threads}");
Console.WriteLine($"  Время: {optimalThreads.avgTime:F2} мс");
Console.WriteLine($"  Выигрыш: {optimalThreads.percentFaster:F1}%\n");

Console.WriteLine("=== Итоговое сравнение ===\n");

double finalSingleTime = singleTimeForStep;
double finalMultiTime = optimalThreads.avgTime;
double finalSpeedup = finalSingleTime / finalMultiTime;
double finalPercentFaster = optimalThreads.percentFaster;

Console.WriteLine($"Шаг: {FormatStep(optimalStep.step)}");
Console.WriteLine($"Потоков: {optimalThreads.threads}");
Console.WriteLine($"Однопоточное время: {finalSingleTime:F2} мс");
Console.WriteLine($"Многопоточное время: {finalMultiTime:F2} мс");
Console.WriteLine($"Ускорение: {finalSpeedup:F2}x");
Console.WriteLine($"Выигрыш: {finalPercentFaster:F1}%");

var plot2 = new Plot();
plot2.Title("Сравнение: Однопоточный vs Многопоточный");
plot2.YLabel("Время (мс)");

var barValues = new double[] { finalSingleTime, finalMultiTime };
var barLabels = new string[] { "Однопоточный", $"Многопоточный\n({optimalThreads.threads} потоков)" };
var bars = plot2.Add.Bars(barValues);
bars.LegendText = "Время выполнения";

for (int i = 0; i < barValues.Length; i++)
{
    var text = plot2.Add.Text($"{barValues[i]:F2} мс", i, barValues[i] + barValues.Max() * 0.02);
    text.Alignment = ScottPlot.Alignment.UpperCenter;
}
var plot1 = new Plot();
plot1.Title($"Время вычисления vs Число потоков (шаг {optimalStep.step:F0e})");
plot1.XLabel("Время (мс)");
plot1.YLabel("Число потоков");

var threadYs = threadResults.Select(r => (double)r.threads).ToArray();
var threadXs = threadResults.Select(r => r.avgTime).ToArray();

var scatter = plot1.Add.Scatter(threadXs, threadYs);
scatter.LegendText = "Многопоточное время";
scatter.MarkerSize = 8;

var vLine = plot1.Add.VerticalLine(finalSingleTime);
vLine.LineColor = ScottPlot.Color.FromHex("#FF0000");
vLine.LineWidth = 2;
vLine.LegendText = "Однопоточное время";

var marker = plot1.Add.Marker(finalMultiTime, optimalThreads.threads, 
    ScottPlot.MarkerShape.FilledCircle, 12, ScottPlot.Color.FromHex("#00AA00"));
marker.LegendText = $"Оптимум: {optimalThreads.threads} потоков";

plot1.ShowLegend();

var fn1 = "plot_thread_performance.png";
plot1.SavePng(fn1, 800, 600);
display(HTML($"<img src='{fn1}?t={DateTime.Now.Ticks}' width='700'/>"));

plot2.Axes.Bottom.SetTicks(new double[] { 0, 1 }, barLabels);
plot2.ShowLegend();

var fn2 = "plot_comparison.png";
plot2.SavePng(fn2, 800, 600);
display(HTML($"<img src='{fn2}?t={DateTime.Now.Ticks}' width='700'/>"));

var reportPath = "report_task15.txt";
var sb = new System.Text.StringBuilder();

sb.AppendLine($"Отчёт по задаче 15: Оптимальные параметры интегрирования");
sb.AppendLine($"Дата: {DateTime.Now}");
sb.AppendLine();
sb.AppendLine($"Функция: sin(x) на отрезке [{-100}, {100}]");
sb.AppendLine($"Требуемая точность: {targetPrecision:F0e}");
sb.AppendLine($"Требуемый выигрыш многопотока: >= 15%");
sb.AppendLine();
sb.AppendLine($"1. Оптимальный шаг: {FormatStep(optimalStep.step)}");
sb.AppendLine($"   - Погрешность: {optimalStep.error:F6}");
sb.AppendLine($"   - Однопоточное время: {optimalStep.singleTime:F2} мс");
sb.AppendLine($"   - Многопоточное время (4 потока): {optimalStep.multiTime:F2} мс");
sb.AppendLine($"   - Выигрыш: {optimalStep.percentFaster:F1}%");
sb.AppendLine();
sb.AppendLine($"2. Оптимальное число потоков: {optimalThreads.threads}");
sb.AppendLine($"   - Время выполнения: {finalMultiTime:F2} мс");
sb.AppendLine($"   - Выигрыш: {finalPercentFaster:F1}%");
sb.AppendLine();
sb.AppendLine($"3. Итоговое сравнение:");
sb.AppendLine($"   - Однопоточное время: {finalSingleTime:F2} мс");
sb.AppendLine($"   - Многопоточное время: {finalMultiTime:F2} мс");
sb.AppendLine($"   - Ускорение: {finalSpeedup:F2}x");
sb.AppendLine($"   - Выигрыш: {finalPercentFaster:F1}%");

File.WriteAllText(reportPath, sb.ToString());

Installed Packages ScottPlot, 5.0.54 xunit, 2.9.2

=== Определение оптимального шага ===

Шаг:   1e-1 | Точность:   0.000000 | Один поток:     0.05 мс | Много:     0.64 мс | Выигрыш: -1200.0%
Шаг:   1e-2 | Точность:   0.000000 | Один поток:     0.67 мс | Много:     0.59 мс | Выигрыш:   12.1%
Шаг:   1e-3 | Точность:   0.000000 | Один поток:     5.71 мс | Много:     2.54 мс | Выигрыш:   55.5%
Шаг:   1e-4 | Точность:   0.000000 | Один поток:    63.68 мс | Много:    24.51 мс | Выигрыш:   61.5%
Шаг:   1e-5 | Точность:   0.000000 | Один поток:   533.27 мс | Много:   204.65 мс | Выигрыш:   61.6%
Шаг:   1e-6 | Точность:   0.000000 | Один поток:  5209.45 мс | Много:  2136.89 мс | Выигрыш:   59.0%

Оптимальный шаг: 1e-3
  Погрешность: 0.000000
  Однопоточное время: 5.71 мс
  Многопоточное время: 2.54 мс
  Выигрыш: 55.5%

=== Определение оптимального числа потоков ===

Потоков:  1 | Время:     6.32 мс | Выигрыш:    1.9%
Потоков:  2 | Время:     3.90 мс | Выигрыш:   39.4%
Потоков:  3 | Время:    10.92 мс | Выигрыш:  -69.5%
Потоков:  4 | Время:    